# Assignment 4 Solution: Retrieval-Augmented Generation (RAG) on SEC 10-K Filings

## Learning Objectives
1. **Build** a FAISS vector index over the SEC 10-K corpus.
2. **Implement** three answering strategies: prompt-only, RAG, and parametric.
3. **Evaluate** all three on a 3×3 question matrix.
4. **Interpret** grounding scores and answer quality.

## Prerequisites
- OpenAI API key (or Gemini) stored as environment variable `OPENAI_API_KEY`.
- `M01_A_sol_artifacts/chunks.csv` from Assignment 1.

> **Good Practice:** FAISS index building is expensive for large corpora.
> Save `faiss_index.bin` and `embeddings.npy` immediately after construction.

In [ ]:
# ── Cell 1: Install and import ────────────────────────────────────────────────
import subprocess, sys
for pkg in ["faiss-cpu", "openai", "tiktoken", "sentence-transformers"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

sns.set_theme(style="whitegrid"); SEED = 42; np.random.seed(SEED)

# ── API key ── (set in Colab Secrets or as environment variable)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client initialized ✓")

In [ ]:
# ── Cell 2: Load artifacts ────────────────────────────────────────────────────
A1_ARTIFACTS = "../M01_A_sol_artifacts"
ARTIFACTS    = "./M05_A_sol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

chunks_df = pd.read_csv(f"{A1_ARTIFACTS}/chunks.csv")
print(f"Chunks loaded: {len(chunks_df):,}")

# Work with up to 3000 chunks to control API cost
if len(chunks_df) > 3000:
    chunks_df = chunks_df.sample(3000, random_state=SEED).reset_index(drop=True)
    print(f"Subsampled to {len(chunks_df):,} chunks for cost control")

## Part 1 — Embed the Corpus and Build a FAISS Index

### Concept: Dense Retrieval with FAISS

**FAISS** (Facebook AI Similarity Search) is a library for efficient
similarity search in high-dimensional spaces.

We use `IndexFlatL2` — an exact L2 (Euclidean) search over all vectors:

$$d(\mathbf{q}, \mathbf{x}_i) = \|\mathbf{q} - \mathbf{x}_i\|_2^2$$

For a query embedding $\mathbf{q}$, FAISS returns the $k$ stored vectors
with smallest L2 distance — equivalent to cosine similarity when vectors
are L2-normalized.

**Embedding model:** `all-MiniLM-L6-v2` (384-d, fast, high-quality).

In [ ]:
# ── Cell 3: Generate embeddings ───────────────────────────────────────────────
EMB_PATH = f"{ARTIFACTS}/embeddings.npy"

if os.path.exists(EMB_PATH):
    print("Loading cached embeddings …")
    embeddings = np.load(EMB_PATH).astype("float32")
else:
    print("Generating embeddings with sentence-transformers …")
    encoder    = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = encoder.encode(
        chunks_df["text"].tolist(),
        batch_size=64, show_progress_bar=True,
        convert_to_numpy=True
    ).astype("float32")
    np.save(EMB_PATH, embeddings)
    print(f"Saved embeddings.npy  shape={embeddings.shape}")

# L2-normalize for cosine similarity
faiss.normalize_L2(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# ── Cell 4: Build FAISS index ─────────────────────────────────────────────────
FAISS_PATH = f"{ARTIFACTS}/faiss_index.bin"

if os.path.exists(FAISS_PATH):
    print("Loading cached FAISS index …")
    index = faiss.read_index(FAISS_PATH)
else:
    DIM   = embeddings.shape[1]
    index = faiss.IndexFlatL2(DIM)
    index.add(embeddings)
    faiss.write_index(index, FAISS_PATH)
    print(f"Built and saved faiss_index.bin  ({index.ntotal:,} vectors)")

print(f"Index has {index.ntotal:,} vectors")

## Part 2 — Three Answering Methods

We implement three strategies for answering business questions about
NVIDIA and AMD:

| # | Method | Description |
|---|---|---|
| 1 | **Prompt-only** | Provide the question to the LLM with no context. Tests parametric knowledge. |
| 2 | **RAG** | Retrieve top-5 relevant chunks from FAISS, include in prompt. Tests grounded retrieval. |
| 3 | **Pretrain-answer** | Include a manually curated 2-sentence company summary (like a brief encyclopedia entry). |

**Why compare these?**
- Prompt-only measures what the LLM already knows (may hallucinate or be outdated).
- RAG grounds the answer in the actual filings (more trustworthy for compliance/audit).
- Pretrain provides a controlled baseline with known context quality.

In [ ]:
# ── Cell 5: Load encoder for RAG retrieval ────────────────────────────────────
if "encoder" not in dir():
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Encoder ready for retrieval ✓")

In [ ]:
# ── Cell 6: Three answering functions ─────────────────────────────────────────
MODEL = "gpt-4o-mini"  # cost-efficient; swap for gpt-4o for higher quality

def prompt_only(question: str) -> str:
    """Answer using only the LLM's parametric (pre-training) knowledge."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "You are a financial analyst. Answer concisely in 2-3 sentences."},
            {"role": "user", "content": question},
        ],
        max_tokens=200,
    )
    return resp.choices[0].message.content.strip()


def rag_answer(question: str, top_k: int = 5) -> str:
    """Retrieve top_k chunks from FAISS and ground the answer."""
    q_emb = encoder.encode([question], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    _, idxs = index.search(q_emb, top_k)
    context_chunks = [chunks_df.iloc[i]["text"] for i in idxs[0]]
    context = "\n\n---\n\n".join(context_chunks)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": (
                 "You are a financial analyst. Use ONLY the provided context to answer. "
                 "If the answer is not in the context, say 'Not found in filings.'"
             )},
            {"role": "user",
             "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        max_tokens=300,
    )
    return resp.choices[0].message.content.strip()


COMPANY_SUMMARIES = {
    "NVDA": (
        "NVIDIA Corporation (NVDA) is a semiconductor company specializing in "
        "GPU-accelerated computing, AI infrastructure, and data-center platforms. "
        "Its flagship products include the H100/H200 Tensor Core GPUs and the "
        "CUDA software ecosystem."
    ),
    "AMD": (
        "Advanced Micro Devices (AMD) designs CPUs and GPUs for PCs, servers, "
        "and embedded systems. Key products include the EPYC server CPU line and "
        "Instinct GPU accelerators for AI/HPC workloads."
    ),
}

def pretrain_answer(question: str, ticker: str = "NVDA") -> str:
    """Answer with a curated company summary as context (parametric + curation)."""
    summary = COMPANY_SUMMARIES.get(ticker, "")
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "You are a financial analyst. Answer in 2-3 sentences."},
            {"role": "user",
             "content": f"Company background: {summary}\n\nQuestion: {question}"},
        ],
        max_tokens=200,
    )
    return resp.choices[0].message.content.strip()


print("Three answering functions defined ✓")

## Part 3 — 3×3 Question Matrix

We test **3 question types × 3 companies/conditions** = 9 combinations.

Question types:
1. **Strategy** — What is [company]'s primary competitive advantage?
2. **Risk** — What are the main risks identified in [company]'s recent 10-K?
3. **Outlook** — What does [company]'s management say about future revenue growth?

In [ ]:
# ── Cell 7: Run 3×3 question matrix ───────────────────────────────────────────
QUESTIONS = {
    "strategy_nvda": "What is NVIDIA's primary competitive advantage according to its 10-K filings?",
    "strategy_amd":  "What is AMD's primary competitive advantage according to its 10-K filings?",
    "strategy_comp": "How does NVIDIA's competitive positioning differ from AMD's in the GPU market?",
    "risk_nvda":     "What are the top risk factors disclosed in NVIDIA's most recent 10-K?",
    "risk_amd":      "What are the top risk factors disclosed in AMD's most recent 10-K?",
    "risk_comp":     "How do NVIDIA's and AMD's disclosed risks differ?",
    "outlook_nvda":  "What does NVIDIA's management say about future revenue growth prospects?",
    "outlook_amd":   "What does AMD's management say about future revenue growth prospects?",
    "outlook_comp":  "Compare NVIDIA's and AMD's management guidance on future revenue.",
}

results = []
for q_id, question in QUESTIONS.items():
    print(f"\n[{q_id}]")
    ticker = "NVDA" if "nvda" in q_id else "AMD"

    print("  → prompt_only …", end=" ")
    ans_po  = prompt_only(question); time.sleep(0.5)
    print("done")

    print("  → rag_answer …", end=" ")
    ans_rag = rag_answer(question); time.sleep(0.5)
    print("done")

    print("  → pretrain_answer …", end=" ")
    ans_pre = pretrain_answer(question, ticker=ticker); time.sleep(0.5)
    print("done")

    results.append({
        "q_id": q_id, "question": question,
        "prompt_only": ans_po,
        "rag": ans_rag,
        "pretrain": ans_pre,
    })

answers_df = pd.DataFrame(results)
answers_df.to_csv(f"{ARTIFACTS}/comparison_answers.csv", index=False)
print("\nSaved comparison_answers.csv")
answers_df[["q_id","rag"]].head()

## Part 4 — Grounding Score and Evaluation

**Grounding score** measures how much of the answer text appears verbatim
(or near-verbatim) in the retrieved chunks.

$$\text{Grounding}(a, C) = \frac{|\text{bigrams}(a) \cap \text{bigrams}(C)|}{|\text{bigrams}(a)|}$$

Where $a$ is the answer and $C$ is the concatenated context chunks.
This is a simplified n-gram overlap metric; production systems use
entailment models (e.g., TrueTeacher, VECTARA).

In [ ]:
# ── Cell 8: Compute grounding scores ─────────────────────────────────────────
def get_bigrams(text: str) -> set:
    words = text.lower().split()
    return {(words[i], words[i+1]) for i in range(len(words)-1)}

def grounding_score(answer: str, context_chunks: list) -> float:
    context_text = " ".join(context_chunks)
    ans_bg = get_bigrams(answer)
    ctx_bg = get_bigrams(context_text)
    if not ans_bg:
        return 0.0
    return len(ans_bg & ctx_bg) / len(ans_bg)

eval_rows = []
for _, row in answers_df.iterrows():
    # Retrieve context for this question
    q_emb = encoder.encode([row["question"]], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    _, idxs = index.search(q_emb, 5)
    context = [chunks_df.iloc[i]["text"] for i in idxs[0]]

    for method in ["prompt_only", "rag", "pretrain"]:
        gs = grounding_score(row[method], context)
        eval_rows.append({
            "q_id": row["q_id"], "method": method,
            "grounding_score": round(gs, 4),
            "answer_length": len(row[method].split()),
        })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(f"{ARTIFACTS}/eval_df.csv", index=False)
print("Saved eval_df.csv")
eval_df.head(9)

In [ ]:
# ── Cell 9: Evaluation heatmap — grounding scores by question × method ────────
pivot = eval_df.pivot(index="q_id", columns="method", values="grounding_score")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, vmin=0, vmax=0.5)
ax.set_title("Grounding Score (Bigram Overlap) — 9 Questions × 3 Methods")
ax.set_xlabel("Answering Method"); ax.set_ylabel("Question ID")
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/grounding_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved grounding_heatmap.png")

In [ ]:
# ── Cell 10: Average grounding and length per method ─────────────────────────
summary = eval_df.groupby("method").agg(
    mean_grounding=("grounding_score", "mean"),
    mean_length=("answer_length", "mean")
).reset_index().sort_values("mean_grounding", ascending=False)
summary.to_csv(f"{ARTIFACTS}/method_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pal = {"rag":"#1f78b4","pretrain":"#33a02c","prompt_only":"#e31a1c"}
sns.barplot(data=summary, x="method", y="mean_grounding", palette=pal, ax=axes[0])
axes[0].set_title("Mean Grounding Score by Method"); axes[0].set_ylim(0, 0.6)
axes[0].set_xlabel("Method"); axes[0].set_ylabel("Mean Bigram Overlap")

sns.barplot(data=summary, x="method", y="mean_length", palette=pal, ax=axes[1])
axes[1].set_title("Mean Answer Length (words) by Method")
axes[1].set_xlabel("Method"); axes[1].set_ylabel("Words")

plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/method_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved method_summary.csv and method_summary.png")

## Assignment Questions — Model Answers

### Q1 — Which method produces the highest grounding score, and why?

**Model Answer:**
The **RAG method** achieves the highest grounding score because the retrieved
chunks are directly injected into the prompt — the LLM is instructed to answer
only from that context, so its output necessarily includes verbatim and
near-verbatim phrases from the filings. Prompt-only answers draw on parametric
memory (may include out-of-date or hallucinated information not present in
any filing) and therefore show near-zero bigram overlap with the corpus.

### Q2 — When does prompt-only outperform RAG?

**Model Answer:**
Prompt-only can outperform RAG on questions that:
- Require **synthesis** across many documents (the top-5 FAISS chunks may miss relevant context).
- Involve **world knowledge** not present in the filings (e.g., macroeconomic context).
- Test **reasoning** rather than retrieval (e.g., "Why would NVDA's GPU monopoly erode?").
RAG is weakest when the question is semantically distant from the stored chunks
(retrieval failure) or when the LLM ignores the provided context.

### Q3 — How would you improve the RAG pipeline for production use?

**Model Answer:**
1. **Better chunking** — Use semantic chunking (e.g., by paragraph/item) instead of fixed word counts.
2. **Hybrid retrieval** — Combine FAISS dense retrieval with BM25 sparse retrieval (reciprocal rank fusion).
3. **Reranker** — Add a cross-encoder reranker (e.g., `ms-marco-MiniLM-L-12-v2`) to re-score the top-20 FAISS hits before passing top-5 to the LLM.
4. **Grounding evaluator** — Replace bigram overlap with an NLI-based entailment model.
5. **Answer faithfulness** — Use RAGAS or LLM-as-judge to evaluate factual consistency.

### Q4 — Business Recommendation

**Model Answer:**
RAG dramatically reduces hallucination risk in financial analysis compared to
prompt-only LLM queries. For an investment research team analyzing SEC filings,
a RAG pipeline anchored to official 10-K text provides **auditable, citable answers**
— the retrieved chunks act as footnotes. This is critical for compliance:
a fund manager can show regulators exactly which filing passage supported each
investment thesis generated by the AI system. The recommended architecture for
production: FAISS index updated at each 10-K filing date, with a nightly
re-embedding pipeline using `all-MiniLM-L6-v2` or a finance-domain encoder.

---
### Artifact Summary
| File | Description |
|---|---|
| `embeddings.npy` | Sentence-transformer embeddings |
| `faiss_index.bin` | FAISS IndexFlatL2 |
| `comparison_answers.csv` | All 9 questions × 3 methods answers |
| `eval_df.csv` | Grounding scores per question × method |
| `grounding_heatmap.png` | 9×3 heatmap of grounding scores |
| `method_summary.png` | Mean grounding + length by method |